# Evaluation on Large Images with Patchification

This notebook demonstrates how to:
1. Load trained models
2. Patch large images into smaller tiles
3. Run predictions on patches
4. Reconstruct full-size predictions
5. Visualize and evaluate results

In [1]:
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import torch
from pathlib import Path
import json

from utils.patchify import (
    predict_large_image,
    patchify_image,
    unpatchify_image
)
from models.model_factory import ModelFactory
from cilia_utils.utils import normalize_channel
from tqdm.notebook import tqdm

## 1. Configuration

In [2]:
# Model configuration
MODEL_CHECKPOINT_ch1 = '../outputs_07Jan25_12-34-00/c1_unet_tiny/best_model.pth'  # Update this path
MODEL_CHECKPOINT_ch2 = '../outputs_07Jan25_12-34-00/c2_unet_tiny/best_model.pth'  # Update this path
ARCHITECTURE = 'unet_tiny'
# CHANNEL_MODE = 'c2_only'  # or 'c1_only'
C1_IDX = 2
C2_IDX = 3
DUAL_MODE = 'overlay'

# Inference configuration
PATCH_SIZE = 128  # Must match training image size
OVERLAP = 48      # Overlap between patches
BATCH_SIZE = 8
THRESHOLD = 0.5
BLEND_MODE = 'average'  # 'average', 'max', or 'first'

# Device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Using device: cuda


## 2. Load Model

In [3]:
def load_model(checkpoint_path, architecture, device='cuda'):
    """Load trained model from checkpoint."""
    model = ModelFactory.create_model(
        architecture,
        num_classes=1,
        pretrained=False
    )
    
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    print(f"Loaded model from: {checkpoint_path}")
    print(f"Epoch: {checkpoint.get('epoch', 'N/A')}")
    print(f"Metrics: {checkpoint.get('metrics', 'N/A')}")
    
    return model

model_ch1 = load_model(MODEL_CHECKPOINT_ch1, ARCHITECTURE, DEVICE)
model_ch2 = load_model(MODEL_CHECKPOINT_ch2, ARCHITECTURE, DEVICE)

Loaded model from: ../outputs_07Jan25_12-34-00/c1_unet_tiny/best_model.pth
Epoch: 73
Metrics: {'loss': 0.17193135246634483, 'iou': 0.755072608590126}
Loaded model from: ../outputs_07Jan25_12-34-00/c2_unet_tiny/best_model.pth
Epoch: 90
Metrics: {'loss': 0.13879333157092333, 'iou': 0.7455189079046249}


In [4]:
# Source - https://stackoverflow.com/a
# Posted by Fábio Perez, modified by community. See post 'Timeline' for change history
# Retrieved 2026-01-07, License - CC BY-SA 4.0

pytorch_total_params = sum(p.numel() for p in model_ch1.parameters())
print(f"Total model parameters: {pytorch_total_params / 1000000:.2f}M")  # Print last 7 digits for brevity

Total model parameters: 1.08M


## 3. Load Test Image(s)

In [5]:
import tifffile as tf
from pathlib import Path
import numpy as np
import os
ROOT = Path(
    '/group/jug/aman/Cilia_Datasets/extracted/'
    'W19 - 2025_Pkhd1_cells/'
    'W19 - 2025_Pkhd1 cells'
)

all_slices = []
slice_index = []  # metadata for reconstruction

print("Scanning folders...")

for cropped_dir in ROOT.rglob('Cropped original image'):
    if not cropped_dir.is_dir():
        continue

    print(f"Found: {cropped_dir}")

    for tif_path in cropped_dir.glob('*.tif'):
        img = tf.imread(tif_path)

        # Normalize to (Z, C, Y, X)
        if img.ndim == 3:           # (C, Y, X)
            img = img[np.newaxis, ...]
        elif img.ndim != 4:
            raise ValueError(f"Unexpected shape {img.shape} in {tif_path}")

        Z, C, Y, X = img.shape

        for z in range(Z):
            all_slices.append(img[z])  # (C, Y, X)
            slice_index.append({
                "path": tif_path,
                "z": z,
                "Z": Z
            })

all_slices = np.stack(all_slices, axis=0)  # (N_total, C, Y, X)
print(f"Total 2D slices collected: {all_slices.shape[0]}")

Scanning folders...
Found: /group/jug/aman/Cilia_Datasets/extracted/W19 - 2025_Pkhd1_cells/W19 - 2025_Pkhd1 cells/CCDC92, Y-tub, Arl13b, DAPI/Pkhd1 KO/Cropped original image
Found: /group/jug/aman/Cilia_Datasets/extracted/W19 - 2025_Pkhd1_cells/W19 - 2025_Pkhd1 cells/CCDC92, Y-tub, Arl13b, DAPI/Pkhd1 CTRL/Cropped original image
Found: /group/jug/aman/Cilia_Datasets/extracted/W19 - 2025_Pkhd1_cells/W19 - 2025_Pkhd1 cells/ZDHHC5, Y-tub, Arl13, DAPI/Pkhd1 KO/Cropped original image
Found: /group/jug/aman/Cilia_Datasets/extracted/W19 - 2025_Pkhd1_cells/W19 - 2025_Pkhd1 cells/ZDHHC5, Y-tub, Arl13, DAPI/Pkhd1 CTRL/Cropped original image
Total 2D slices collected: 510


### Example Image 

## 4. Prepare Image for Inference

In [ ]:
def prepare_image_for_inference(image, channel_mode='c2_only', c1_idx=0, c2_idx=1, dual_mode='average'):
    """
    Prepare multichannel image for model inference.
    
    Args:
        image: (C, H, W) multichannel image
        channel_mode: 'c1_only', 'c2_only', or 'dual'
        c1_idx: Channel 0 index
        c2_idx: Channel 1 index
        dual_mode: 'average', 'zeros', or 'overlay'
    
    Returns:
        RGB image (3, H, W) ready for model
    """
    if channel_mode == 'c1_only':
        c1 = image[c1_idx]
        c1_norm = normalize_channel(c1)
        rgb = np.stack([c1_norm] * 3, axis=0)
    
    elif channel_mode == 'c2_only':
        c2 = image[c2_idx]
        c2_norm = normalize_channel(c2)
        rgb = np.stack([c2_norm] * 3, axis=0)
    
    # IMPORTANT: Keep pixel values in [0, 255] range to match training!
    # The model was trained on uint8 images converted to float (not normalized to [0,1])
    return (rgb * 255).astype(np.uint8).astype(np.float32)



In [7]:
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display

H = all_slices.shape[-1]  # image height

# -------------------------
# Widgets
# -------------------------

# Slice slider
slice_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(all_slices) - 1,
    step=1,
    description='Slice:',
    continuous_update=False
)

# Crop range sliders
x_slider = widgets.IntRangeSlider(
    value=[400, 600],
    min=0,
    max=H,
    step=1,
    description='X',
    continuous_update=False
)

y_slider = widgets.IntRangeSlider(
    value=[400, 600],
    min=0,
    max=H,
    step=1,
    description='Y',
    continuous_update=False
)

# Channel toggle
channel_toggle = widgets.ToggleButtons(
    options=[('Channel 1', 'ch1'), ('Channel 2', 'ch2')],
    value='ch1',
    description='Channel:'
)

# -------------------------
# Visualization function
# -------------------------

def show_prediction(test_idx, x_slider, y_slider, channel):
    # Unpack range sliders
    x1, x2 = x_slider
    y1, y2 = y_slider

    # Safety checks
    x1, x2 = max(0, x1), min(H, x2)
    y1, y2 = max(0, y1), min(H, y2)

    if x2 <= x1 or y2 <= y1:
        return

    crop_x = slice(x1, x2)
    crop_y = slice(y1, y2)

    raw_image = all_slices[test_idx]

    # -------------------------
    # Channel selection
    # -------------------------
    if channel == 'ch1':
        prepared_image = prepare_image_for_inference(
            raw_image,
            'c1_only',
            C1_IDX,
            C2_IDX,
            DUAL_MODE
        )
        model = model_ch1
    else:
        prepared_image = prepare_image_for_inference(
            raw_image,
            'c2_only',
            C1_IDX,
            C2_IDX,
            DUAL_MODE
        )
        model = model_ch2

    # -------------------------
    # Prediction
    # -------------------------
    pred_mask, pred_binary = predict_large_image(
        model=model,
        image=prepared_image,
        patch_size=PATCH_SIZE,
        overlap=32,
        batch_size=BATCH_SIZE,
        blend_mode=BLEND_MODE,
        device=DEVICE,
        threshold=THRESHOLD
    )

    # -------------------------
    # Plotting
    # -------------------------
    plt.figure(figsize=(12, 4))

    # Input image
    plt.subplot(1, 3, 1)
    plt.imshow(prepared_image[0][crop_y, crop_x], cmap='gray')
    plt.title(f'Input Image ({channel.upper()})')
    plt.axis('off')

    # Overlay
    plt.subplot(1, 3, 2)
    plt.imshow(prepared_image[0][crop_y, crop_x], cmap='gray')

    magenta_mask = np.zeros((*pred_binary[crop_y, crop_x].shape, 3))
    magenta_mask[..., 0] = pred_binary[crop_y, crop_x]
    magenta_mask[..., 2] = pred_binary[crop_y, crop_x]

    plt.imshow(magenta_mask, alpha=0.5)
    plt.title('Overlayed Prediction')
    plt.axis('off')

    # Binary mask
    plt.subplot(1, 3, 3)
    plt.imshow(pred_binary[crop_y, crop_x], cmap='gray')
    plt.title('Predicted Mask')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

# -------------------------
# Interactive display
# -------------------------

widgets.interact(
    show_prediction,
    test_idx=slice_slider,
    x_slider=x_slider,
    y_slider=y_slider,
    channel=channel_toggle
)


interactive(children=(IntSlider(value=0, continuous_update=False, description='Slice:', max=509), IntRangeSlid…

<function __main__.show_prediction(test_idx, x_slider, y_slider, channel)>

In [8]:
all_pred_masks = []
all_pred_binary = []

for i, raw_slice in tqdm(enumerate(all_slices), total=len(all_slices)):
    prepared_image = prepare_image_for_inference(
        raw_slice,                 # (C, H, W)
        CHANNEL_MODE,
        C1_IDX,
        C2_IDX,
        DUAL_MODE
    )                               # (3, H, W)

    pred_mask, pred_binary = predict_large_image(
        model=model,
        image=prepared_image,
        patch_size=PATCH_SIZE,
        overlap=32,
        batch_size=BATCH_SIZE,
        blend_mode=BLEND_MODE,
        device=DEVICE,
        threshold=THRESHOLD
    )

    all_pred_masks.append(pred_mask)       # (H, W)
    all_pred_binary.append(pred_binary)    # (H, W)

    if i % 10 == 0:
        print(f"Processed slice {i+1}/{len(all_slices)}")
all_pred_masks = np.stack(all_pred_masks, axis=0)     # (N_total, H, W)
all_pred_binary = np.stack(all_pred_binary, axis=0)  # (N_total, H, W)
print("All slices processed.")

  0%|          | 0/510 [00:00<?, ?it/s]

NameError: name 'CHANNEL_MODE' is not defined

In [ ]:
pred_dict_mask = {}
pred_dict_binary = {}

for info, mask, binary in zip(slice_index, all_pred_masks, all_pred_binary):
    path = info["path"]
    z = info["z"]
    Z = info["Z"]

    if path not in pred_dict_mask:
        pred_dict_mask[path] = [None] * Z
        pred_dict_binary[path] = [None] * Z

    pred_dict_mask[path][z] = mask
    pred_dict_binary[path][z] = binary


In [ ]:
for path in pred_dict_mask:
    pred_dict_mask[path] = np.stack(pred_dict_mask[path], axis=0)
    pred_dict_binary[path] = np.stack(pred_dict_binary[path], axis=0)

    print(
        path.name,
        pred_dict_mask[path].shape,
        pred_dict_binary[path].shape
    )


IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_3_original cropped.tif (15, 675, 675) (15, 675, 675)
IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_4_original cropped.tif (19, 675, 675) (19, 675, 675)
IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_7_original cropped.tif (15, 675, 675) (15, 675, 675)
IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_5_original cropped.tif (16, 675, 675) (16, 675, 675)
IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_1_original cropped.tif (16, 675, 675) (16, 675, 675)
IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_8_original cropped.tif (17, 675, 675) (17, 675, 675)
IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_6_original cropped.tif (16, 675, 675) (16, 675, 675)
IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_2_original cropped.tif (18, 675, 675) (18, 675, 675)
IMCD3 Pkhd CTRL - CCDC92, ARL13B, Y-TUB, DAPI_2_original cropped.tif (15, 675, 675) (15, 675, 675)
IMCD3 Pkhd CTRL - CCDC92, ARL13B, Y-TUB, DAPI_6_original cropped.tif (14, 675, 675) (14, 675, 675)
IMCD3 Pkhd CTRL - CCDC92, 

In [ ]:
PRED_ROOT = "/group/jug/aman/cilia"Predictions"
PRED_ROOT.mkdir(exist_ok=True)

for tif_path in pred_dict_mask:
    # Get path relative to ROOT
    rel_path = tif_path.relative_to(ROOT)

    # Remove filename, keep folder structure
    rel_dir = rel_path.parent

    # Create mirrored directory inside Predictions/
    out_dir = PRED_ROOT / rel_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    # Write outputs
    tf.imwrite(
        out_dir / tif_path.name.replace(".tif", "_mask.tif"),
        pred_dict_mask[tif_path].astype(np.float32)
    )

    tf.imwrite(
        out_dir / tif_path.name.replace(".tif", "_binary.tif"),
        pred_dict_binary[tif_path].astype(np.uint8)
    )
    print(f"Saved predictions for {tif_path.name} to {out_dir}")

Saved predictions for IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_3_original cropped.tif to /group/jug/aman/Cilia_Datasets/extracted/W19 - 2025_Pkhd1_cells/W19 - 2025_Pkhd1 cells/Predictions/CCDC92, Y-tub, Arl13b, DAPI/Pkhd1 KO/Cropped original image
Saved predictions for IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_4_original cropped.tif to /group/jug/aman/Cilia_Datasets/extracted/W19 - 2025_Pkhd1_cells/W19 - 2025_Pkhd1 cells/Predictions/CCDC92, Y-tub, Arl13b, DAPI/Pkhd1 KO/Cropped original image
Saved predictions for IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_7_original cropped.tif to /group/jug/aman/Cilia_Datasets/extracted/W19 - 2025_Pkhd1_cells/W19 - 2025_Pkhd1 cells/Predictions/CCDC92, Y-tub, Arl13b, DAPI/Pkhd1 KO/Cropped original image
Saved predictions for IMCD3 Pkhd KO - CCDC92, ARL13B, Y-TUB, DAPI_5_original cropped.tif to /group/jug/aman/Cilia_Datasets/extracted/W19 - 2025_Pkhd1_cells/W19 - 2025_Pkhd1 cells/Predictions/CCDC92, Y-tub, Arl13b, DAPI/Pkhd1 KO/Cropped original im

## 6. Visualize Results

In [ ]:
from matplotlib import patches
def visualize_dual_predictions(
    original_img, pred_ch1, pred_ch2,
    z_slice=None, figsize=(20, 16),
    patch_size=128, seed=None, show_patch_box=True,
    transpose=False  # NEW
):
    """
    Visualize original channels, predictions, and a random zoomed patch.
    
    Args:
        transpose: if True, swaps last two axes (X/Y) for display
    """

    if seed is not None:
        np.random.seed(seed)

    # Handle 4D input
    if original_img.ndim == 4:
        if z_slice is None:
            z_slice = 0
        original_img = original_img[z_slice]
        pred_ch1 = pred_ch1[z_slice] if pred_ch1.ndim == 3 else pred_ch1
        pred_ch2 = pred_ch2[z_slice] if pred_ch2.ndim == 3 else pred_ch2

    # Transpose if needed
    if transpose:
        original_img = original_img[..., ::-1].copy() if original_img.ndim == 3 else original_img.transpose(0,2,1)
        pred_ch1 = pred_ch1.T
        pred_ch2 = pred_ch2.T

    H, W = pred_ch1.shape

    # --------------------
    # Random patch
    # --------------------
    ph = pw = patch_size
    y0 = np.random.randint(0, H - ph)
    x0 = np.random.randint(0, W - pw)

    # --------------------
    # Normalize originals
    # --------------------
    img_ch1 = original_img[C1_IDX]
    img_ch2 = original_img[C2_IDX]

    img_ch1_norm = (img_ch1 - img_ch1.min()) / (img_ch1.max() - img_ch1.min() + 1e-8)
    img_ch2_norm = (img_ch2 - img_ch2.min()) / (img_ch2.max() - img_ch2.min() + 1e-8)

    overlay = np.stack([img_ch1_norm, img_ch2_norm, np.zeros_like(img_ch1_norm)], axis=2)
    combined_pred = np.stack([pred_ch1, pred_ch2, np.zeros_like(pred_ch1)], axis=2)

    # --------------------
    # Figure
    # --------------------
    fig, axes = plt.subplots(3, 3, figsize=figsize)
    fig.suptitle(f'Dual Channel Predictions (Z={z_slice})', fontsize=16)

    # Row 1: full images
    axes[0, 0].imshow(img_ch1_norm, cmap='viridis')
    axes[0, 0].set_title(f'Original Channel {C1_IDX}')
    axes[0, 0].axis('off')

    axes[0, 1].imshow(img_ch2_norm, cmap='viridis')
    axes[0, 1].set_title(f'Original Channel {C2_IDX}')
    axes[0, 1].axis('off')

    axes[0, 2].imshow(overlay)
    axes[0, 2].set_title('Overlay (R=Ch1, G=Ch2)')
    axes[0, 2].axis('off')

    # Row 2: full predictions
    axes[1, 0].imshow(pred_ch1, cmap='gray')
    axes[1, 0].set_title('Prediction – Channel 1')
    axes[1, 0].axis('off')

    axes[1, 1].imshow(pred_ch2, cmap='gray')
    axes[1, 1].set_title('Prediction – Channel 2')
    axes[1, 1].axis('off')

    axes[1, 2].imshow(combined_pred)
    axes[1, 2].set_title('Predictions Overlay')
    axes[1, 2].axis('off')

    # Row 3: zoomed patch
    axes[2, 0].imshow(img_ch1_norm[y0:y0+ph, x0:x0+pw], cmap='viridis')
    axes[2, 0].set_title('Zoom – Orig Ch1')
    axes[2, 0].axis('off')

    axes[2, 1].imshow(img_ch2_norm[y0:y0+ph, x0:x0+pw], cmap='viridis')
    axes[2, 1].set_title('Zoom – Orig Ch2')
    axes[2, 1].axis('off')

    axes[2, 2].imshow(combined_pred[y0:y0+ph, x0:x0+pw])
    axes[2, 2].set_title('Zoom – Predictions')
    axes[2, 2].axis('off')

    # --------------------
    # Draw patch box
    # --------------------
    if show_patch_box:
        for ax in axes[0]:
            rect = patches.Rectangle(
                (x0, y0), pw, ph,
                linewidth=2, edgecolor='yellow', facecolor='none'
            )
            ax.add_patch(rect)

    plt.tight_layout()
    plt.show()


In [ ]:
import ipywidgets as widgets
from IPython.display import display

def show_prediction(test_idx):
    visualize_dual_predictions(
        original_img=all_slices[test_idx],
        pred_ch1=all_pred_binary[test_idx],
        pred_ch2=all_pred_binary[test_idx],
        figsize=(18, 10),
        patch_size=128,
        transpose=True,   # swap axes for correct orientation
        seed=42
)


slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(all_slices) - 1,
    step=1,
    description='Slice:',
    continuous_update=False
)

widgets.interact(show_prediction, test_idx=slider)


interactive(children=(IntSlider(value=0, continuous_update=False, description='Slice:', max=509), Output()), _…

<function __main__.show_prediction(test_idx)>